# SOC Agent — Standard of Care Analysis

In [1]:
import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))

from agent_prompt_functions import get_soc_agent_prompt
from data_utils import filterSOCLOT, formatSOCLotDF

print(f'CT_ROOT : {CT_ROOT}')

CT_ROOT : /Users/rajeevkulkarni/ml-explorations/clinical-trial-optimizer


In [2]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

API key set


In [3]:
DATA_FILE = CT_ROOT / 'data' / 'raw' /'rwd_lot.xlsx'

sheets = pd.read_excel(DATA_FILE, sheet_name=None)
print(f'Sheets found: {list(sheets.keys())}')

# Adjust sheet name to match your workbook
soc_raw = sheets['soc_lot']

filtered_rows, topN_per_line = filterSOCLOT(soc_raw, pct=0.03, N=5)
soc_df = formatSOCLotDF(topN_per_line)

print(f'Rows after filterSOCLOT (top-N): {len(topN_per_line)}')
print(soc_df[:500])

Sheets found: ['soc_lot', 'lot_c1', 'lot_c2']
Rows after filterSOCLOT (top-N): 15
| Regimen                              | Line Number   |   Count | % (Per Line)   |
|:-------------------------------------|:--------------|--------:|:---------------|
| osimertinib                          | L1R1          |     921 | 61.4%          |
| osimertinib+carboplatin+pemetrexed   | L1R2          |     180 | 12.0%          |
| gefitinib                            | L1R3          |     120 | 8.0%           |
| erlotinib                            | L1R4          |      90 | 6.0%         


In [4]:
# Paste I/E criteria here
ie_criteria = """
Inclusion Criteria:
- Participant must have at least 1 measurable lesion, according to Response Evaluation Criteria in Solid Tumors (RECIST) version 1.1, that has not been previously irradiated
- Participant must have histologically or cytologically confirmed, locally advanced or metastatic, non-squamous non-small cell lung cancer (NSCLC), characterized at or after the time of locally advanced or metastatic disease diagnosis by either epidermal growth factor receptor (EGFR) Exon 19del or Exon 21 L858R mutation
- A participant with a history of brain metastases must have had all lesions treated as clinically indicated (that is, no current indication for further definitive local therapy). Any definitive local therapy to brain metastases must have been completed at least 14 days prior to randomization and the participant can be receiving no greater than10 milligrams (mg) prednisone or equivalent daily for the treatment of intracranial disease
- Participant must have Eastern Cooperative Oncology Group (ECOG) status of 0 or 1
- Any toxicities from prior systemic anticancer therapy must have resolved to National Cancer Institute Common Terminology Criteria for Adverse Events (NCI CTCAE) Version 5.0 Grade 1 or baseline level (except for alopecia [any grade], Grade <= 2 peripheral neuropathy, or Grade <= 2 hypothyroidism stable on hormone replacement)
- A participant of childbearing potential must have a negative serum pregnancy test at screening and within 72 hours of the first dose of study treatment and must agree to further serum or urine pregnancy tests during the study
- Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second- line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy. Participants who received either neoadjuvant and/or adjuvant treatment of any type are eligible if progression to locally advanced or metastatic disease occurred at least 12 months after the last dose of such therapy and then the participant progressed on or after osimertinib in the locally advanced or metastatic setting. Treatment with osimertinib must be discontinued at least 8 days (4 half-lives) prior to randomization (that is last dose no later than Day -8)

Exclusion Criteria:
- Participant received radiotherapy for palliative treatment of NSCLC less than 14 days prior to randomization
- Participant with symptomatic or progressive brain metastases
- Participant has history of or current evidence of leptomeningeal disease, or participant has spinal cord compression not definitively treated with surgery or radiation
- Participant has known small cell transformation
- Participant has a medical history of interstitial lung disease (ILD), including drug-induced ILD or radiation pneumonitis
- Participant has a history of clinically significant cardiovascular disease including, but not limited to diagnosis of deep vein thrombosis or pulmonary embolism within 4 weeks prior to randomization; myocardial infarction; unstable angina; stroke; transient ischemic attack; coronary/peripheral artery bypass graft; or acute coronary syndrome. Participant has a significant genetic predisposition to venous thromboembolic events. Participant has a prior history of venous thromboembolic events and is not on appropriate therapeutic anticoagulation as per National Comprehensive Cancer Network or local guidelines"
"""

print(f'Length of I/E criteria: {len(ie_criteria)} characters')

Length of I/E criteria: 3585 characters


In [5]:
soc_prompt = get_soc_agent_prompt(ie_criteria, soc_df)
print(f'Prompt length: {len(soc_prompt)} chars')

Prompt length: 18968 chars


In [6]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls the Claude API
client = anthropic.Anthropic()

response = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=4096,
    messages=[{'role': 'user', 'content': soc_prompt}],
)

display(Markdown(response.content[0].text))

## Step 0–2 Internal Analysis (Not Emitted)

*(Processing internally: disease = locally advanced or metastatic EGFR-mutant [Exon 19del or L858R] non-squamous NSCLC; trial target line = post-osimertinib, i.e., second-line or later after osimertinib monotherapy; treatment-related criteria identified and mapped to dataset.)*

---

## Output

### 1. Gap Analysis Table

| Type | Original Criterion | Gap Type | Queriable | Expected Increase | Explanation | Recommendation |
|------|--------------------|----------|-----------|-------------------|-------------|----------------|
| Inclusion | Participant must have progressed on or after **osimertinib monotherapy** as the most recent line of treatment. Osimertinib must have been administered as either the **first-line** treatment for locally advanced or metastatic disease **or in the second-line setting after prior treatment with first- or second-generation EGFR TKI as a monotherapy**. | Major | Yes | ~12% | The criterion explicitly allows osimertinib in L1 (**osimertinib L1R1: 61.4%**) or in L2 after a 1st/2nd-gen EGFR TKI (**osimertinib L2R1: 41.6%**). However, the dataset shows **osimertinib+carboplatin+pemetrexed at L1R2 (12.0%)** — a combination regimen where osimertinib was not given as monotherapy. Patients who received this combination in L1 would be excluded by the monotherapy restriction, despite representing a meaningful real-world segment. Additionally, **gefitinib (L1R3: 8.0%), erlotinib (L1R4: 6.0%), and afatinib (L1R5: 5.0%)** confirm robust real-world 1st/2nd-gen TKI → osimertinib sequencing, which is already captured by the criterion. The monotherapy restriction on osimertinib creates a gap for the 12% who received osimertinib-based combinations in L1. | Exclude participants who received **osimertinib in combination with other systemic agents (e.g., osimertinib+carboplatin+pemetrexed)** as first-line or second-line systemic therapy for locally advanced or metastatic NSCLC, unless osimertinib was subsequently administered as monotherapy prior to enrollment. |
| Inclusion | Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as **first-line** treatment OR in **second-line after first- or second-generation EGFR TKI monotherapy**. | Minor | Yes | ~8% | The criterion permits osimertinib in L2 only after **first- or second-generation EGFR TKI as monotherapy**. Real-world data shows patients who received **carboplatin+pemetrexed (L2R2: 25.1%)**, **carboplatin+pemetrexed+bevacizumab (L2R3: 10.0%)**, **carboplatin+pemetrexed+pembrolizumab (L2R4: 7.5%)**, or **docetaxel (L2R5: 6.0%)** in L2 — these patients may have had a non-TKI L2 regimen before osimertinib in L3 (not shown in top 5 L3 as osimertinib). The criterion does not explicitly address osimertinib given in the **third-line or later** setting after non-TKI L2 therapy. However, the L3 data does not show osimertinib as a top-5 regimen, limiting the quantifiable impact. The gap is **Minor** given the dataset does not surface a substantial osimertinib L3 population. | Already captured by existing I/E Criteria |
| Exclusion | Participants who received **neoadjuvant and/or adjuvant treatment of any type** are eligible only if progression to locally advanced or metastatic disease occurred at least 12 months after the last dose and then progressed on or after osimertinib in the locally advanced/metastatic setting. | No Gap | No | N/A | The dataset summary does not include adjuvant or neoadjuvant treatment data; this criterion cannot be evaluated against dataset. Criterion is clinically appropriate and well-specified. | Already captured by existing I/E Criteria |

---

### 2. Post-Processing: Consolidated Suggestions

**Based on Original Criterion:**
*"Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second-line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy."*

- **Suggested Addition 1 (Gap Type: Major):** Exclude participants who received **osimertinib in combination with other systemic agents (e.g., osimertinib+carboplatin+pemetrexed)** as first-line or second-line systemic therapy for locally advanced or metastatic NSCLC, unless osimertinib was subsequently administered as monotherapy prior to enrollment.

**Updated Combined Criterion:**
Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second-line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy. OR Exclude participants who received osimertinib in combination with other systemic agents (e.g., osimertinib+carboplatin+pemetrexed) as first-line or second-line systemic therapy for locally advanced or metastatic NSCLC, unless osimertinib was subsequently administered as monotherapy prior to enrollment.

---

### 3. Therapeutic Context Summary

**Disease & Biomarker Context:**
The trial targets patients with histologically/cytologically confirmed **locally advanced or metastatic, non-squamous NSCLC** harboring **EGFR Exon 19 deletion or Exon 21 L858R mutation**, who have progressed on osimertinib monotherapy. This represents a post-osimertinib second-line (or later) setting in a biomarker-selected population.

**Line of Therapy Alignment:**
The trial's primary prior-treatment requirement centers on osimertinib monotherapy in L1 or L2. Real-world data confirms that **osimertinib is the dominant L1 regimen at 61.4% (L1R1, n=921)** and a major L2 regimen at **41.6% (L2R1, n=312)**. The 1st/2nd-generation EGFR TKI → osimertinib (L2) sequence is well-supported by **gefitinib (L1R3: 8.0%), erlotinib (L1R4: 6.0%), and afatinib (L1R5: 5.0%)** in L1, confirming the real-world validity of the permitted sequencing paths described in the inclusion criterion. The trial's target enrollment population is therefore well-aligned with real-world practice for the osimertinib monotherapy pathway.

**Key Observations:**
- **osimertinib+carboplatin+pemetrexed at L1R2 (12.0%, n=180)** is a clinically significant real-world regimen that the current monotherapy restriction would exclude. This is the most impactful gap identified.
- The **1st/2nd-gen TKI → osimertinib L2** sequence is robustly represented in the dataset, validating that the inclusion criterion's second permitted pathway captures a meaningful real-world population.
- L2 chemotherapy regimens (**carboplatin+pemetrexed: 25.1%; with bevacizumab: 10.0%; with pembrolizumab: 7.5%**) suggest some patients may have had complex multi-line histories before reaching osimertinib, but the dataset does not surface osimertinib as a top-5 L3 regimen, limiting actionable conclusions for deeper-line osimertinib use.
- Adjuvant/neoadjuvant treatment data is absent from the dataset, so the 12-month washout eligibility clause for prior adjuvant therapy cannot be quantified.

**Primary Recommendation:**
The most impactful protocol amendment would be to explicitly address the **osimertinib+carboplatin+pemetrexed combination (L1R2: 12.0%)** — either by clarifying whether patients who received this combination and then progressed are eligible (if osimertinib component alone is considered sufficient) or by explicitly excluding them if the monotherapy intent is strict. Resolving this ambiguity could affect eligibility for approximately **12% of the real-world L1 patient population** and is critical for enrollment accuracy and protocol consistency.